In [ ]:
# hide
# no-output
from IPython.utils.capture import capture_output
with capture_output():
    %pip install -q plotly anywidget

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
import icm_plotly
from icm_plotly import RED, BLUE, GOLD, IRON, TEAL, STEEL

Drag $B$. The red curve is the call overhead and the blue curve is the
peak memory, both counted in samples. The three rows of the table are
marked along the top: small blocks pay in calls, and large blocks pay in
memory.

In [ ]:
# hide
# autorun
N, M = 44100, 3                     # one second of audio, three unit generators
CALL = 100                          # suppose one call costs about 100 samples of work
B0 = 441                            # starting parameter: the chapter's block size

def calls(B, M=M, N=N):
    return M * int(np.ceil(N / B))

def overhead(B, CALL=CALL, calls=calls):
    return CALL * calls(B)

def memory(B, M=M):
    return M * B

# every block size that splits one second evenly: 81 of them, 1 to 44100
DIVISORS = [d for d in range(1, N + 1) if N % d == 0]
B_GRID = np.array(DIVISORS)
OVH = np.array([overhead(b) for b in B_GRID])
MEM = np.array([memory(b) for b in B_GRID])
NAMED_B = [1, 441, N]
NAMED_TEXT = ["sample-by-sample", "B = 441", "ugen-by-ugen"]
Y_TOP = 3e7

def figure():
    fig = go.Figure()
    fig.add_scatter(x=B_GRID, y=OVH, mode="lines", line=dict(color=RED, width=2.4))
    fig.add_scatter(x=B_GRID, y=MEM, mode="lines", line=dict(color=BLUE, width=2.4))
    fig.add_scatter(x=[B0, B0], y=[1, Y_TOP], mode="lines",
                    line=dict(color=GOLD, width=1.6, dash="dash"))
    fig.add_scatter(x=[B0, B0], y=[overhead(B0), memory(B0)], mode="markers",
                    marker=dict(color=GOLD, size=11, line=dict(color="white", width=2)))
    fig.add_scatter(x=NAMED_B, y=[Y_TOP] * 3, mode="markers+text", text=NAMED_TEXT,
                    textposition=["bottom right", "bottom center", "bottom left"],
                    marker=dict(color=IRON, size=7, symbol="triangle-down"),
                    textfont=dict(size=12))
    fig.add_scatter(x=[3, 2500], y=[1.5e7, 2.2e2], mode="text",
                    text=["overhead", "memory"], textposition="middle right",
                    textfont=dict(color=[RED, BLUE], size=13))
    fig.update_xaxes(type="log", range=[np.log10(0.8), np.log10(60000)],
                     title_text="Block size B (samples)", fixedrange=True)
    fig.update_yaxes(type="log", range=[0, np.log10(6e7)],
                     title_text="Cost (samples)", fixedrange=True)
    return fig

def controls(fig):
    b = widgets.SelectionSlider(description="Block size B",
                                options=[(f"{d:,}", d) for d in DIVISORS],
                                value=B0)
    readout = widgets.HTML()

    # the defaults snapshot the helpers; the page's notebooks share one kernel
    def update(B, calls=calls, overhead=overhead, memory=memory,
               readout=readout):
        with fig.batch_update():
            fig.data[2].x = [B, B]
            fig.data[3].x = [B, B]
            fig.data[3].y = [overhead(B), memory(B)]
        readout.value = (f"<span style='font-size:0.9em'>{calls(B):,} calls, "
                         f"about {overhead(B):,} samples of overhead "
                         f"&nbsp;·&nbsp; peak memory {memory(B):,} samples"
                         f"</span>")

    widgets.interactive_output(update, {"B": b})
    return widgets.VBox([b, readout])

icm_plotly.show(figure, controls)